# grasp_verify: did we actually pick the piece up? verify and retry

Library notebook for `gambit_arm`. The README targets 55-60 % pick-and-place success on the
first attempt and 95 % after automatic verify-and-retry. This notebook defines the checks
that decide "the piece is in the gripper" / "the piece landed on its square", and the retry
loop that uses them. Measuring the rates is run_pick_place_trials.ipynb.

Milestone: *Hardware and execution* (gripper fingers shaped to the pieces, with grasp
verification; 95 % after verify-and-retry).

**TODO**
- [ ] grasp signals data class and a rule-based verdict (gripper position + current)
- [ ] retry loop with max attempts and a deterministic search pattern of small offsets
- [ ] camera check: piece gone from the source square after lift, present and centered on
      the target after place (needs gambit_vision board reading; see `camera_check`)
- [ ] calibrate thresholds per piece type on the real arm (safety.yaml `gripper` section)
- [ ] knocked-neighbor detection after a failed grasp (camera diff of the 8 neighbors)
- [ ] log every attempt (signals, verdict, offset) for the trials notebook
- [ ] write the self-checks at the bottom as asserts

In [ ]:
from dataclasses import dataclass, field
import numpy as np

%run ./pick_place.ipynb

GRIP = SAFETY["gripper"]

## Signals and verdict

After commanding `gripper_closed`, a gripper with a piece between the jaws **stalls early**:
its measured position stays well above the commanded closed value, and it draws a holding
current. An empty gripper closes fully (position below `closed_empty_threshold`) and draws
little current. Any one signal can lie (a piece caught on one fingertip still stalls), so the
camera check is the final word when available.

In [ ]:
@dataclass
class GraspSignals:
    gripper_pos: float              # rad, measured after closing
    gripper_current_ma: float = None
    camera_says_gone: bool = None   # source square empty after lift (None = not checked)

@dataclass
class Verdict:
    ok: bool
    reasons: list = field(default_factory=list)

def verify_grasp(sig: GraspSignals, cfg=GRIP) -> Verdict:
    """Rule-based verdict: is the piece in the gripper?"""
    # Design:
    # - Position rule: measured gripper position below gripper.closed_empty_threshold means
    #   the jaws closed fully -> nothing between them.
    # - Current rule (if measured): holding current must lie in gripper.grasp_current_ma
    #   [lo, hi]; too low = empty, too high = jammed / pinching the board.
    # - Camera (if checked): camera_says_gone is False -> piece still on the source square.
    # - ok only if no rule fires; keep every reason (for the attempt log and debugging).
    # Note None means "not measured" and must not fail the check.
    # TODO: implement
    raise NotImplementedError

def camera_check(square, expect_occupied: bool) -> bool:
    """True if the square's occupancy (from the camera) matches `expect_occupied`."""
    # Design: read the board with gambit_vision (classifier on the square crop); for placement
    # also check the piece center is within tolerance of the square center
    # (square_map.offset_from_center).
    # TODO: implement once gambit_vision board reading works
    raise NotImplementedError("camera_check needs gambit_vision board reading")

## Retry policy

On a failed grasp: open, lift back to hover, and try again at a slightly shifted grasp point
(the piece may be off-center on its square). Offsets follow a fixed pattern (center, then
+-x, +-y at `offset_step`) so trials are reproducible. After `max_attempts` give up and ask
the human; never keep poking a piece that may already be knocked over.

In [ ]:
@dataclass
class RetryPolicy:
    max_attempts: int = 3
    offset_step: float = 0.003      # m. tune on the real arm
    pattern: tuple = ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1))

    def offset(self, attempt: int) -> np.ndarray:
        """(3,) grasp-point offset in m for attempt number `attempt` (0 = first)."""
        # Design: pattern entry (wrapping around) times offset_step, z = 0. Deterministic so
        # trials are reproducible.
        # TODO: implement
        raise NotImplementedError

@dataclass
class AttemptLog:
    attempt: int
    offset: np.ndarray
    verdict: Verdict

def pick_with_retry(backend, square, kind: str, policy=RetryPolicy(), sense=None):
    """Pick `kind` from `square`, verifying after each attempt.
    backend: a pick_place Backend. sense(backend) -> GraspSignals (defaults to the gripper
    position only). Returns (success, [AttemptLog])."""
    # Design:
    # - For each attempt up to policy.max_attempts: build pick(square, kind), shift only the
    #   descend and grasp waypoints by policy.offset(attempt) (the hover stays put), plan and
    #   execute, sense, verify_grasp, log.
    # - On success return immediately; on failure open and go back to hover before retrying
    #   (never drag a half-grasped piece sideways).
    # - After max_attempts give up (False): the caller asks the human.
    # TODO: implement
    raise NotImplementedError

def success_rates(trials):
    """trials: list of lists of AttemptLog (one list per pick). Returns
    (first-attempt rate, after-retry rate) for the README targets."""
    # Design: first-attempt = fraction whose first log is ok; after-retry = fraction whose
    # last log is ok. Empty attempt lists count as failures.
    # TODO: implement
    raise NotImplementedError

## Self-check (plan: fake backend with an empty gripper for the first k attempts)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - Fake backend: subclass DryRunBackend, count executions containing a "grasp" target, and
#   make gripper_position() report ~0.03 rad (closed, empty) for the first `fail_first`
#   grasps and ~0.4 rad (stalled on a piece) afterwards.
# - verify_grasp: (0.4 rad, 200 mA) ok; 0.02 rad not ok; 5000 mA not ok; camera_says_gone=False not ok
# - pick_with_retry with fail_first=1: success, 2 logs, first not ok, second offset (0.003, 0, 0)
# - fail_first=5 with max_attempts=3: failure after exactly 3 logs
# - success_rates([[ok], [fail, ok]]) == (0.5, 1.0)
# Example shape of a test:
# assert not verify_grasp(GraspSignals(0.02)).ok
print("Self-check: not written yet (see plan above)")
print("grasp_verify OK")